# NB_hospital_daily_summary

Builds `dbx_caresync.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, patient/doctor counts, abnormal/critical rates, and day-over-day trend metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE dbx_caresync.gold.hospital_daily_summary
AS
WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date                                                    AS summary_date,
    COUNT(*)                                                        AS total_tests,
    COUNT(DISTINCT lr.patient_id)                                    AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                     AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
  FROM dbx_caresync.silver.lab_results AS lr
  GROUP BY lr.hospital_id, lr.test_date
),
with_rates AS (
  SELECT
    da.hospital_id,
    h.hospital_name,
    da.summary_date,
    c.day_name,
    c.is_weekend,
    da.total_tests,
    da.unique_patients_tested,
    da.active_doctor_count,
    da.abnormal_test_count,
    da.critical_test_count,
    ROUND(da.abnormal_test_count / da.total_tests * 100, 2)         AS abnormal_rate_pct,
    ROUND(da.critical_test_count / da.total_tests * 100, 2)         AS critical_rate_pct
  FROM daily_agg                                   AS da
  LEFT JOIN dbx_caresync.silver.hospitals           AS h
         ON da.hospital_id = h.hospital_id
  LEFT JOIN dbx_caresync.silver.calendar            AS c
         ON c.date_key = CAST(date_format(da.summary_date, 'yyyyMMdd') AS INT)
)
SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)  AS prev_day_total_tests,
  ROUND(
    CASE
      WHEN LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) IS NULL
        OR LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) = 0
      THEN NULL
      ELSE (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
           / LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) * 100
    END, 2
  )                                                                       AS test_volume_change_pct
FROM with_rates
ORDER BY hospital_id, summary_date;

In [0]:
updated_df = spark.table("dbx_caresync.gold.hospital_daily_summary")
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))